# Triton Notes

## flash-attention

我们梳理一下flash-attention的逻辑：

flash-attention中是按照O_block来分配线程块的

In [4]:
import torch
import triton
import triton.language as tl

In [5]:
@triton.jit
def add_kernel(x_ptr,
               y_ptr,
               output_ptr,
               n_elements,
               BLOCK_SIZE:tl.constexpr):

    pid=tl.program_id(axis=0)

    offsets=pid*BLOCK_SIZE+tl.arange(0,BLOCK_SIZE)

    mask=offsets<n_elements

    x=tl.load(x_ptr+offsets,mask=mask)          #这里offsets是list，x_ptr和列表中每个元素依次相加，得到的结果将再返回一个列表.此处的加法就是C++中的指针位移操作
    y=tl.load(y_ptr+offsets,mask=mask)          #load就会遍历返回的列表中的每一个指针然后取出对应的元素，最后返回给x或者y，两者也都是一个列表

    output=x+y

    tl.store(output_ptr+offsets,output,mask=mask)

In [6]:
def add(x:torch.Tensor,y:torch.Tensor):

    output=torch.empty_like(x)
    assert x.is_cuda and y.is_cuda and output.is_cuda

    n_elements=output.numel()

    grid=lambda meta:(tl.cdiv(n_elements,meta['BLOCK_SIZE']),)

    add_kernel[grid](x,y,output,n_elements,BLOCK_SIZE=1024)

    return output

Triton 的自动打包机制：
当 grid 被定义为一个可调用对象（如代码中的 lambda meta: ...）时，Triton 框架在底层会自动拦截传递给 add_kernel 的关键字参数（即 BLOCK_SIZE=1024），并将它们打包成一个字典。
这个字典随后会被作为 meta 参数传递给 grid 这个 lambda 函数。因此，在 lambda 函数内部读取 meta['BLOCK_SIZE'] 时，就能获取到值 1024。

## Reduction(归约)算子

<img src="./img/1.png" width=70%>

```
m_ij=tl.maximum(m_i,tl.max(QK_block,axis=1))
```

<img src="./img/2.png" width=70%>

## 我们为什么要保存LSE? 

<img src="./img/3.png" width=70%>

$l_{i}$ 是我们计算的归一化因子，也就是每一行所有数据的指数和，具体为：
$$
l_{i}=\sum_{n=0}^{m} \mathbf{e} ^ {x_{n}}
$$

我们在代码中将m_i赋成了m_i+log(li)

```
lse=m_i+tl.math.log(l_i)
m_ptr=M+index_batch_head*SEQ_LEN+offsets_q
tl.store(m_ptr,lse)

```

这样之后就只需要exp(x_i-m_i)即可，不需要额外再除以l_i

## `tl.make_block_ptr()`的使用

<img src="./img/4.png" width=70%>

## Autograd in Pytorch

### 标量多元函数、向量多元函数与Jacobian矩阵

#### 标量多元函数对向量(一维张量)求导

$$
f:\mathbb{R}^{N}\rightarrow\mathbb{R}
$$
$$
f([x_1,x_2,\ldots,x_N])=y
$$
$$
\vec{x}=[x_1,x_2,\ldots,x_N]
$$
$$
f(\vec{x})=y
$$
$$
\frac{\partial f}{\partial\vec{x}}
=
\begin{bmatrix}
\frac{\partial y}{\partial x_1}, &
\frac{\partial y}{\partial x_2}, &
\cdots &
\frac{\partial y}{\partial x_N}
\end{bmatrix}
$$



#### 标量多元函数对矩阵(二维张量)求导

$$
f:\mathbb{R}^{N\times M}\rightarrow\mathbb{R}
$$
$$
f\left(
\begin{bmatrix}
x_{11} & x_{12} & \cdots & x_{1M}\\
\vdots & \vdots & \ddots & \vdots\\
x_{N1} & x_{N2} & \cdots & x_{NM}
\end{bmatrix}
\right)=y
$$
$$
\mathbf{X}=
\begin{bmatrix}
x_{11} & x_{12} & \cdots & x_{1M}\\
\vdots & \vdots & \ddots & \vdots\\
x_{N1} & x_{N2} & \cdots & x_{NM}
\end{bmatrix}
$$
$$
f(\mathbf{X})=y
$$
$$
\frac{\partial f}{\partial\mathbf{X}}
=
\begin{bmatrix}
\frac{\partial y}{\partial x_{11}} &
\frac{\partial y}{\partial x_{12}} &
\cdots &
\frac{\partial y}{\partial x_{1M}}\\
\vdots & \vdots & \ddots & \vdots\\
\frac{\partial y}{\partial x_{N1}} &
\frac{\partial y}{\partial x_{N2}} &
\cdots &
\frac{\partial y}{\partial x_{NM}}
\end{bmatrix}
$$

#### 向量多元函数对矩阵(二维张量)求导

$$
f:\mathbb{R}^{N\times M}\rightarrow\mathbb{R}^{k}
$$
$$
f\left(
\begin{bmatrix}
x_{11} & x_{12} & \cdots & x_{1M}\\
\vdots & \vdots & \ddots & \vdots\\
x_{N1} & x_{N2} & \cdots & x_{NM}
\end{bmatrix}
\right)
=
\begin{bmatrix}
y_1\\
y_2\\
\vdots\\
y_k
\end{bmatrix}
$$
$$
f(\mathbf{X})=\vec{y}
$$
$$
\frac{\partial f}{\partial\mathbf{X}}
=
\begin{bmatrix}
\begin{bmatrix}
\frac{\partial y_1}{\partial x_{11}} &
\frac{\partial y_1}{\partial x_{12}} &
\cdots &
\frac{\partial y_1}{\partial x_{1M}}\\
\vdots & \vdots & \ddots & \vdots\\
\frac{\partial y_1}{\partial x_{N1}} &
\frac{\partial y_1}{\partial x_{N2}} &
\cdots &
\frac{\partial y_1}{\partial x_{NM}}
\end{bmatrix}
\\[1em]
\begin{bmatrix}
\frac{\partial y_2}{\partial x_{11}} &
\frac{\partial y_2}{\partial x_{12}} &
\cdots &
\frac{\partial y_2}{\partial x_{1M}}\\
\vdots & \vdots & \ddots & \vdots\\
\frac{\partial y_2}{\partial x_{N1}} &
\frac{\partial y_2}{\partial x_{N2}} &
\cdots &
\frac{\partial y_2}{\partial x_{NM}}
\end{bmatrix}
\\[1em]
\vdots
\\[1em]
\begin{bmatrix}
\frac{\partial y_k}{\partial x_{11}} &
\frac{\partial y_k}{\partial x_{12}} &
\cdots &
\frac{\partial y_k}{\partial x_{1M}}\\
\vdots & \vdots & \ddots & \vdots\\
\frac{\partial y_k}{\partial x_{N1}} &
\frac{\partial y_k}{\partial x_{N2}} &
\cdots &
\frac{\partial y_k}{\partial x_{NM}}
\end{bmatrix}
\end{bmatrix}
$$

我们称
$$
\frac{\partial f}{\partial\mathbf{X}}
=
\begin{bmatrix}
\frac{\partial y}{\partial x_{11}} &
\frac{\partial y}{\partial x_{12}} &
\cdots &
\frac{\partial y}{\partial x_{1M}}\\
\vdots & \vdots & \ddots & \vdots\\
\frac{\partial y}{\partial x_{N1}} &
\frac{\partial y}{\partial x_{N2}} &
\cdots &
\frac{\partial y}{\partial x_{NM}}
\end{bmatrix}
$$
为$Jacobian$矩阵。

称
$$
\frac{\partial f}{\partial\mathbf{X}}
=
\begin{bmatrix}
\begin{bmatrix}
\frac{\partial y_1}{\partial x_{11}} &
\frac{\partial y_1}{\partial x_{12}} &
\cdots &
\frac{\partial y_1}{\partial x_{1M}}\\
\vdots & \vdots & \ddots & \vdots\\
\frac{\partial y_1}{\partial x_{N1}} &
\frac{\partial y_1}{\partial x_{N2}} &
\cdots &
\frac{\partial y_1}{\partial x_{NM}}
\end{bmatrix}
\\[1em]
\begin{bmatrix}
\frac{\partial y_2}{\partial x_{11}} &
\frac{\partial y_2}{\partial x_{12}} &
\cdots &
\frac{\partial y_2}{\partial x_{1M}}\\
\vdots & \vdots & \ddots & \vdots\\
\frac{\partial y_2}{\partial x_{N1}} &
\frac{\partial y_2}{\partial x_{N2}} &
\cdots &
\frac{\partial y_2}{\partial x_{NM}}
\end{bmatrix}
\\[1em]
\vdots
\\[1em]
\begin{bmatrix}
\frac{\partial y_k}{\partial x_{11}} &
\frac{\partial y_k}{\partial x_{12}} &
\cdots &
\frac{\partial y_k}{\partial x_{1M}}\\
\vdots & \vdots & \ddots & \vdots\\
\frac{\partial y_k}{\partial x_{N1}} &
\frac{\partial y_k}{\partial x_{N2}} &
\cdots &
\frac{\partial y_k}{\partial x_{NM}}
\end{bmatrix}
\end{bmatrix}
$$

为$Generalized-Jacobian$矩阵

我们可以将梯度视作行数为一(m=1)的$Jacobian$矩阵